In [14]:
import numpy as np

Definition of Acrylic Sheet

In [15]:
resolution = 0.1

x_size = 10
y_size = 10
z_size = 4

x = np.arange(0, x_size, resolution)
y = np.arange(0, y_size, resolution)    
z = np.arange(0, z_size, resolution)

acrylic_charge = np.zeros((len(x), len(y), len(z)))

Building Space Charge

In [16]:
N = 1000
electrons = np.zeros((N, 3))

# Beam-injected depth profile: electrons stop around z_inject with spread sigma_z,
# clipped to stay inside the slab.
z_inject = z_size / 2
sigma_z  = z_size / 8

for i in range(N):
    x_rand = np.random.uniform(0, x_size)
    y_rand = np.random.uniform(0, y_size)
    while True:
        z_rand = np.random.normal(z_inject, sigma_z)
        if 0.0 <= z_rand <= z_size:
            break

    electrons[i] = [x_rand, y_rand, z_rand]

Electron-Native Discharge

The acrylic is empty dielectric except at the $N$ electron sites. A crack only has to decide which electron to reach next — not what the field is doing everywhere in vacuum. We drop the grid entirely.

Tree is a graph: nodes are the nail + every collected electron, edges are straight segments from each collected electron to the tree node it attached to.

At every step,

1. For each still-remaining electron $i$ look up $d_i$, the distance to its nearest tree node.
2. Sample one electron with probability $P_i \propto 1/d_i^{\eta}$. Large $\eta$ → greedy (always the nearest, Prim-like). Small $\eta$ → diffuse, long jumps. $\eta \approx 3$ gives Lichtenberg-ish branching.
3. Connect it to that nearest tree node with one line segment. Remove it from the remaining set.

The grounding BC is implicit: once an electron joins the tree it's at $V=0$ and stops contributing to anyone else's field — so collected electrons silently disappear from the bookkeeping. No Poisson solve, no relaxation.

Cost is $O(N^2)$ total. For $N=1000$ that's under a second.

In [17]:
eta = 3.0

# Nail (grounded electrode) at the centre of the top face (z = 0)
nail = np.array([x_size / 2.0, y_size / 2.0, 0.0])

tree_pos    = [nail]   # list of node positions (cm)
tree_parent = [-1]     # parent index; -1 marks the root (nail)

remaining = np.ones(N, dtype=bool)

# For each electron: distance to its nearest tree node, and the index of that node.
# Maintained incrementally as the tree grows.
nearest_dist = np.linalg.norm(electrons - nail, axis=1)
nearest_node = np.zeros(N, dtype=np.int64)

rng = np.random.default_rng(0)

while remaining.any():
    w = np.zeros(N)
    w[remaining] = 1.0 / (nearest_dist[remaining] ** eta + 1e-30)
    p = w / w.sum()

    pick = rng.choice(N, p=p)
    parent_idx = int(nearest_node[pick])

    new_node_idx = len(tree_pos)
    tree_pos.append(electrons[pick].copy())
    tree_parent.append(parent_idx)
    remaining[pick] = False

    # The new node may now be the closest tree node for some remaining electrons.
    d_new = np.linalg.norm(electrons - electrons[pick], axis=1)
    closer = (d_new < nearest_dist) & remaining
    nearest_dist[closer] = d_new[closer]
    nearest_node[closer] = new_node_idx

tree_pos    = np.array(tree_pos)      # (M, 3), cm
tree_parent = np.array(tree_parent)   # (M,)

# Depth in edges from the nail — handy for colouring
depth = np.zeros(len(tree_pos), dtype=np.int32)
for i in range(1, len(tree_pos)):
    depth[i] = depth[tree_parent[i]] + 1

print(f"Tree: {len(tree_pos)} nodes, max depth {depth.max()} edges from nail")

Tree: 1001 nodes, max depth 32 edges from nail


Interactive 3D View

Branches are the segments joining each collected electron to its parent in the tree. Colour = depth from the nail (number of edges), so the trunk is dark and distant tips are bright.

In [18]:
import plotly.graph_objects as go

# Per-segment vertices with None separators so Plotly draws each edge independently.
parents = tree_parent[1:]
children = np.arange(1, len(tree_pos))
P = tree_pos[parents]
C = tree_pos[children]

edge_x = np.column_stack([P[:, 0], C[:, 0], np.full(len(P), np.nan)]).ravel()
edge_y = np.column_stack([P[:, 1], C[:, 1], np.full(len(P), np.nan)]).ravel()
edge_z = np.column_stack([P[:, 2], C[:, 2], np.full(len(P), np.nan)]).ravel()

fig3d = go.Figure([
    go.Scatter3d(
        x=edge_x, y=edge_y, z=edge_z,
        mode='lines',
        line=dict(color='orange', width=2),
        hoverinfo='skip',
        name='branches',
    ),
    go.Scatter3d(
        x=tree_pos[1:, 0], y=tree_pos[1:, 1], z=tree_pos[1:, 2],
        mode='markers',
        marker=dict(size=2, color=depth[1:], colorscale='Inferno',
                    colorbar=dict(title='depth')),
        name='collected electrons',
        hovertemplate='depth %{marker.color}<extra></extra>',
    ),
    go.Scatter3d(
        x=[nail[0]], y=[nail[1]], z=[nail[2]],
        mode='markers',
        marker=dict(size=7, color='cyan', symbol='diamond'),
        name='nail',
    ),
])
fig3d.update_layout(
    scene=dict(
        xaxis=dict(title='x (cm)', range=[0, x_size]),
        yaxis=dict(title='y (cm)', range=[0, y_size]),
        zaxis=dict(title='z (cm)', range=[0, z_size]),
        aspectmode='data',
    ),
    title=f'Lichtenberg discharge — N={N}, η={eta}',
    margin=dict(l=0, r=0, b=0, t=30),
    width=900, height=700,
)
fig3d.show()